In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

In [2]:
from src.data_loader import VesselDataLoader
from src.visualizer import VesselVisualizer
from src.data_processing import DataProcessor
from src.mission_profiler import MissionProfiler
from src.mission_profiler import ScenarioEvaluator

In [6]:
# Path to your raw telemetry
raw_data_file1 = root_path / "data" / "raw" / "Rotherhithe_voy_179.csv"
raw_data_file2 = root_path / "data" / "raw" / "Wembley_voy_236.csv"


loader1 = VesselDataLoader(raw_data_file1)
loader2 = VesselDataLoader(raw_data_file2)

df1 = DataProcessor(loader1.load_and_clean()).extract_features_and_filter(filter_method='raw')
df2 = DataProcessor(loader2.load_and_clean()).extract_features_and_filter(filter_method='raw')


master_df = pd.concat([df1, df2], ignore_index=False)

profiler = MissionProfiler(speed_threshold=1.0)
labeled_df = profiler.classify_phases(master_df)

brick_registry = profiler.generate_phase_registry(labeled_df, source_file_name="Combined_Sister_Vessels")

global_stats = profiler.extract_global_statistics(labeled_df)

In [7]:
plotter = VesselVisualizer(labeled_df) # (Passes structural check, can plot master_registry directly)
fig_bricks = plotter.plot_brick_space(brick_registry, y_axis_metric='Intensive_L2_Volatility')
fig_bricks.show()
fig_stats = plotter.plot_phase_statistics(global_stats)
fig_stats.show()

In [8]:
evaluator = ScenarioEvaluator(global_stats)

# Define the scenario weights (Targeting exactly 1000 hours for SEC Energy House)
# For this theoretical run, we are bypassing port loads (assuming cold ironing)
test_weights = {
    'Sea_Transit_Laden': 324.0,
    'Sea_Transit_Ballast': 119.0,
    'Sea_Loitering': 161.0,
    'Port_Idle': 217.0,
    'Port_Loading': 51.0,
    'Port_Unloading': 128.0
}

scenario_results = evaluator.evaluate(time_weights_hours=test_weights)

print("\n=== MARINER 1000h Scenario Evaluation ===")
for k, v in scenario_results.items():
    if k == 'Weights_Applied':
        continue
    print(f"{k}: {v:.2f}")


=== MARINER 1000h Scenario Evaluation ===
Scenario_Hours: 1000.00
Expected_Energy_kWh: 570254.46
Expected_Total_H2_Lower_kg: 31117.24
Expected_Total_H2_Upper_kg: 38032.18
Expected_L2_Fatigue_Index: 31341.06
